In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from unidecode import unidecode
import sys, os, csv
from datetime import datetime

src = Path.cwd().parent/"src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

pd.set_option('display.max_columns', None)

from routes.Paths import (
    Ruta_busquedad_contacto,
    Ruta_arbol_tipificacion,
    Ruta_opl,
    Ruta_festivos,
    Ruta_masterskill,
    Ruta_crm,
    Ruta_Comentarios,
    Ruta_local,
    Ruta_global
)

require_columns = {
    "id de contacto",
    "canal",
    "marca de tiempo de inicio",
    "cola",
    "agente",
    "marca de tiempo de desconexion",
    "marca de tiempo de conexion a agente",
    "marca de tiempo en cola",
    "marca de tiempo de inicio de acw",
    "marca de tiempo de fin de acw"
}

delete_dup_columns = ["id de contacto"]

from utils.Processing_OPL import data_opl
from utils.Processing_Arbol import limpiar_df_arbol
from utils.Processing_MasterSkill import limpiar_masterskill

from utils.functions import load_files_from_folder

from utils.Processing_data_comentarios import _transformacion_comentarios
from utils.Data_CRM.a_carga_data import load_files
from utils.Data_CRM.e_final_data_crm import transform_crm

df_contactos = load_files_from_folder(
    Ruta_busquedad_contacto,
    require_columns,
    delete_dup_columns
)
df_contactos = df_contactos[~df_contactos["cola"].isna()]

df_arbol = limpiar_df_arbol(pd.read_excel(Ruta_arbol_tipificacion, sheet_name='Tipificación Vista360'))
df_opl = data_opl(pd.read_excel(Ruta_opl, sheet_name="Combinado"))
df_master = limpiar_masterskill(pd.read_excel(Ruta_masterskill, sheet_name="TB"))
df_festivos = (
    pd.to_datetime( pd.read_csv(Ruta_festivos)["Fecha"], format= "%d/%m/%Y")
    .drop_duplicates()
    .values.astype("datetime64[D]")
)

df_comentarios = _transformacion_comentarios(Ruta_Comentarios)

In [28]:
df_crm = transform_crm(
    load_files(Ruta_crm),
    df_arbol,
    df_festivos,
    df_contactos,
    df_opl,
    df_master,
    df_comentarios)

In [29]:
columnas_requeridas = [
'id de contacto',
'id de flujo del primer contacto',
'id de contacto inicial',
'id del contacto anterior',
'id de contacto siguiente',
'canal',
'cola',
'perfil de enrutamiento',
'numero de telefono del cliente',
'numero de telefono del sistema',
'marca de tiempo de inicio',
'marca de tiempo de desconexion',
'duracion del contacto',
'marca de tiempo de conexion a agente',
'marca temporal programada',
'marca de tiempo en cola',
'marca de tiempo de inicio de acw',
'marca de tiempo de fin de acw',
'duracion de la interaccion del agente',
'numero de esperas',
'metodo de iniciacion',
'motivo de la desconexion',
'subtipo de canal',
'direccion de contacto',
'asunto del correo electronico',
'direccion de correo electronico del sistema',
'direccion de correo electronico del cliente',
'source_file',
'file_date',
'agente'
]

df_contactos = df_contactos[columnas_requeridas]
normalizar_columnas = [
    "canal",
    "cola",
    "perfil de enrutamiento",

]

for col in normalizar_columnas:
    df_contactos[col] = (
        df_contactos[col].astype("string")
        .str.strip()
        .str.title()
        .map(lambda x: unidecode(x) if pd.notna(x) else x)
    )

df_contactos = df_contactos.merge(
    df_crm[[
    "ID_LLAMADA_DE_CONTACTO",
    "NIVEL_1",
    "NIVEL_2",
    "NIVEL_3",
    "NÚMERO_DOCUMENTO_DEL_CLIENTE",
    "NOMBRE_ALIADO",
    "SOLUCION",
    "IdTipificacion",
    "MÉTODO_DE_APROBACIÓN_DE_VALIDACIÓN",
    "CÓDIGO_DE_VALIDACIÓN"
    ]],
    how="left",
    left_on=["id de contacto"],
    right_on=["ID_LLAMADA_DE_CONTACTO"]
). drop(columns=["ID_LLAMADA_DE_CONTACTO"])

df_contactos = df_contactos.merge(
    df_master[[
        "IdMasterSkill",
        "Canal",
        "Colas"
    ]],
    how="left",
    left_on=["canal", "cola"],
    right_on=["Canal", "Colas"]
).drop(columns=["Canal", "Colas"])

df_contactos = df_contactos.merge(
    df_opl[[
        "Id_opl",
        "Correo_contratista"
    ]],
    how="left",
    left_on=["agente"],
    right_on=["Correo_contratista"],
).drop(columns=["Correo_contratista"])

In [ ]:
columnas_datetime =[
    "marca de tiempo de inicio",
    "marca de tiempo de desconexion",
    "marca de tiempo de conexion a agente",
    "marca de tiempo en cola",
    "marca de tiempo de inicio de acw",
    "marca de tiempo de fin de acw",
    "file_date"
    ]

for col in columnas_datetime:
    df_contactos[col] = pd.to_datetime(
                df_contactos[col].astype(str).str.strip(),
                format="%d/%m/%Y %H:%M",
                errors="coerce"
            )
df_contactos["fecha_inicio"] = df_contactos["marca de tiempo de inicio"].dt.date
df_contactos["fecha_fin"] = df_contactos["marca de tiempo de desconexion"].dt.date
df_contactos["llave_documento_fecha"] = (
    df_contactos["NÚMERO_DOCUMENTO_DEL_CLIENTE"].astype(str) + df_contactos["fecha_inicio"].astype(str)
)
df_contactos["llave_recontacto"] = (
    df_contactos["NÚMERO_DOCUMENTO_DEL_CLIENTE"].astype(str) + 
    df_contactos["fecha_inicio"].astype(str) + 
    df_contactos["IdTipificacion"].astype(str)
)

In [31]:
tipificaciones_excluir = {
    "Llamada de Prueba",
    "Interacción de Prueba",
    "Chat de Prueba",
    "Pruebas PCO",
    "Devolución llamada EFECTIVA",
    "Devolución llamada NO EFECTIVA"
}

documentos_excluir = {
    "2222222956",
    "2222222005"
}

In [32]:
df_valido = df_contactos[
    ~df_contactos["metodo de iniciacion"].isin(["Transferir", "Saliente"]) &
    (~df_contactos["NIVEL_3"].isin(tipificaciones_excluir) |
    ~df_contactos["NÚMERO_DOCUMENTO_DEL_CLIENTE"].astype(str).isin(documentos_excluir))
].copy()

In [33]:
df_valido = df_valido.sort_values([
    "llave_documento_fecha", "marca de tiempo de inicio"
])

#calculos rellamadas
df_valido["orden_interaccion"] = (
    df_valido.groupby(
        "llave_documento_fecha")
        .cumcount() + 1
    )

df_valido["conteo_rellamda"] = (
    df_valido.groupby("llave_documento_fecha")["id de contacto"]
    .transform("count")
    )

df_valido["es_rellamada"] = (
    df_valido["orden_interaccion"] > 1
).astype(int)

#calculos recontactos
df_valido["conteo_recontacto"] = (
    df_valido.groupby("llave_recontacto")["id de contacto"]
    .transform("count")
    )

df_valido["orden_recontacto"] = (
    df_valido
    .sort_values("marca de tiempo de inicio")
    .groupby("llave_recontacto")
    .cumcount()
    + 1
    )

df_valido["es_recontacto"] = (
    df_valido["orden_recontacto"] > 1
).astype(int)

In [34]:
df_resumen = df_valido.sort_values(["llave_documento_fecha", "orden_interaccion"]).copy()
df_resumen["es_chat"] = df_resumen["canal"] == "Chat"
df_resumen["es_voz"] = df_resumen["canal"] == "Voz"

df_resumen = df_resumen.groupby("llave_documento_fecha", sort=False).agg(
    primer_contacto = ("canal", "first"),
    tiene_chat = ("es_chat", "any"),
    tiene_voz = ("es_voz", "any")
).reset_index()

df_resumen["voz_a_chat"] = ((df_resumen["primer_contacto"] == "Voz") & df_resumen["tiene_chat"]).astype(int)
df_resumen["chat_a_voz"] = ((df_resumen["primer_contacto"] == "Chat") & df_resumen["tiene_voz"]).astype(int)

df_valido = df_valido.merge(
    df_resumen[["llave_documento_fecha", "voz_a_chat", "chat_a_voz"]],
    on="llave_documento_fecha",
    how="left"
)

In [35]:
df_contactos = df_contactos.merge(
    df_valido[
        ["id de contacto", "orden_interaccion", 
         "conteo_rellamda", "es_rellamada", 
         "conteo_recontacto", "orden_recontacto", 
         "es_recontacto", "voz_a_chat", "chat_a_voz"
         ]],
         on="id de contacto",
         how="left"
)

In [36]:
df_contactos.to_csv(os.path.join(Ruta_global, "05.Data_Contactos.csv"), index=False, encoding="utf-8")
df_contactos.to_csv(os.path.join(Ruta_local, "05.Data_Contactos.csv"), index=False, encoding="utf-8")

In [37]:
df_contactos.isna().sum()

id de contacto                                      0
id de flujo del primer contacto                 52407
id de contacto inicial                         253689
id del contacto anterior                       253689
id de contacto siguiente                        55703
canal                                               0
cola                                                0
perfil de enrutamiento                           1797
numero de telefono del cliente                      0
numero de telefono del sistema                      0
marca de tiempo de inicio                           0
marca de tiempo de desconexion                      0
duracion del contacto                               0
marca de tiempo de conexion a agente             9639
marca temporal programada                      263460
marca de tiempo en cola                         52407
marca de tiempo de inicio de acw                 9639
marca de tiempo de fin de acw                    9639
duracion de la interaccion d